### Define Risk Segmentation Logic
| Risk Level      | Credit Score Range |
| --------------- | ------------------ |
| **High Risk**   | `< 580`            |
| **Medium Risk** | `580 – 699`        |
| **Low Risk**    | `≥ 700`            |


### Connect To SQL Server

In [1]:
import pyodbc
import pandas as pd 
import numpy as np

DRIVER_NAME = 'ODBC Driver 17 for SQL Server'
SERVER_NAME = r'DESKTOP-L3GBMQ5\SQLEXPRESS'
DATABASE_NAME = 'Financial_CaseStudy'

connection_string = (
    f"DRIVER={{{DRIVER_NAME}}};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    f"Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)
cursor = conn.cursor()
print("Connected successfully!")

Connected successfully!


### KPIs segmented by Risk Level

In [2]:
segmented_query = """
WITH RiskSegment AS (
    SELECT *
    ,CASE
        WHEN Credit_Score_Final < 580 
            THEN 'High Risk'
        WHEN Credit_Score_Final BETWEEN 580 AND 669
            THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS Risk_Level
    FROM 
        dbo.dw_loan_analysis
)
SELECT 
    Risk_Level
    ,ROUND(SUM(CASE WHEN Is_Default = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Default_Rate_Percent
    ,ROUND(AVG(Interest_Rate), 2) AS Avg_Interest_Rate
    ,ROUND(AVG(Loan_Amount), 2) AS Avg_Loan_Amount
    ,ROUND(AVG(Annual_Income), 2) AS Avg_Customer_Income
FROM
    RiskSegment
GROUP BY
    Risk_Level
ORDER BY
    CASE Risk_Level
        WHEN 'Low Risk' THEN 1
        WHEN 'Medium Risk' THEN 2
        WHEN 'High Risk' THEN 3
    END;
"""

### Execute the query and load into Pandas 

In [4]:
risk_kpi_df = pd.read_sql(segmented_query, conn)
print('Risk-Level KPIs:')
risk_kpi_df

C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_2100\3617261041.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  risk_kpi_df = pd.read_sql(segmented_query, conn)


Risk-Level KPIs:


,Risk_Level,Default_Rate_Percent,Avg_Interest_Rate,Avg_Loan_Amount,Avg_Customer_Income
0,Low Risk,0.0,11.37,27688.94,87539.22
1,Medium Risk,0.0,11.22,27154.32,79086.23
2,High Risk,0.0,11.31,27635.15,84510.54


### Segment by Age Group or Income

In [5]:
age_query = """
SELECT 
    Customer_Age_Group
    ,ROUND(SUM(CASE WHEN Is_Default = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Default_Rate_Percent
    ,ROUND(AVG(Interest_Rate), 2) AS Avg_Interest_Rate
FROM 
    dbo.dw_loan_analysis
GROUP BY 
    Customer_Age_Group
ORDER BY 
    Customer_Age_Group;
"""

age_kpi_df = pd.read_sql(age_query, conn)
display(age_kpi_df)

C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_2100\3359248404.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  age_kpi_df = pd.read_sql(age_query, conn)


,Customer_Age_Group,Default_Rate_Percent,Avg_Interest_Rate
0,<25,0.0,11.41
1,25-35,0.0,11.14
2,36-50,0.0,11.35
